<img src="images/banner.png" style="width: 100%;">

# Web Scraping with Python

In [ ]:
# !conda install html5lib -y # You might need to instll html5lib for the proper HTML parsing

References:

[1] Mitchell, Ryan. *Web scraping with python.* " O'Reilly Media, Inc.", 2024.

[2] Chapagain, Anish. *Hands-On Web Scraping with Python: Extract quality data from the web using effective Python techniques.* Packt Publishing Ltd, 2023.

[3] Elridge, Taryn. *Web automation and scraping with Python: Hands-on mastery of selenium, beautifulsoup, and real-world data extraction.* 2025.

[4] Requests Documentation - https://requests.readthedocs.io/en/latest/user/quickstart/

[5] Beautiful Soup Documentation - https://beautiful-soup-4.readthedocs.io/en/latest/

[6] Revised and grammar checked using ChatGPT - https://chatgpt.com/

Prepared by: Leodegario Lorenzo II

This notebook introduces the fundamentals of web scraping using Python and the `BeautifulSoup` library.

In a nutshell, **web scraping** is the process of extracting data from websites programmatically. In other words, we’ll use Python to interact with web pages the way a human would through a browser—requesting pages, reading their content, and pulling out the information we care about.

In this demo, we’ll walk through:

1. How to make HTTP requests with Python
2. How to parse and navigate HTML documents with BeautifulSoup
3. Common design patterns and best practices for more advanced web scraping tasks

By the end, you should have a solid fundamentals for how web scraping works and the confidence to build your own scrapers responsibly.

## 0 Web Scraping Prerequisites

### `robots.txt`

Before scraping any website, you should first check the site’s `robots.txt` file. This file indicates which parts of the site the owner allows or disallows for automated access. Reviewing it helps you avoid scraping areas the site explicitly asks bots not to access, and sets a baseline for responsible, respectful scraping practices.

For most of the initial examples in this notebook, we'll be scraping the website from one of our primary reference - [pythonscraping.com](https://pythonscraping.com/). A quick look on its `robots.txt` file indicate that most of the pages are allowed for scraping, except for particular files and login-protected directories:

```
#
# robots.txt
#
# This file is to prevent the crawling and indexing of certain parts
# of your site by web crawlers and spiders run by sites like Yahoo!
# and Google. By telling these "robots" where not to go on your site,
# you save bandwidth and server resources.
#
# This file will be ignored unless it is at the root of your host:
# Used:    http://example.com/robots.txt
# Ignored: http://example.com/site/robots.txt
#
# For more information about the robots.txt standard, see:
# http://www.robotstxt.org/robotstxt.html
#
# For syntax checking, see:
# http://www.frobee.com/robots-txt-check

User-agent: *
Crawl-delay: 10
# Directories
Disallow: /includes/
Disallow: /misc/
Disallow: /modules/
Disallow: /profiles/
Disallow: /scripts/
Disallow: /themes/
# Files
Disallow: /CHANGELOG.txt
Disallow: /cron.php
Disallow: /INSTALL.mysql.txt
Disallow: /INSTALL.pgsql.txt
Disallow: /INSTALL.sqlite.txt
Disallow: /install.php
Disallow: /INSTALL.txt
Disallow: /LICENSE.txt
Disallow: /MAINTAINERS.txt
Disallow: /update.php
Disallow: /UPGRADE.txt
Disallow: /xmlrpc.php
# Paths (clean URLs)
Disallow: /admin/
Disallow: /comment/reply/
Disallow: /filter/tips/
Disallow: /node/add/
Disallow: /search/
Disallow: /user/register/
Disallow: /user/password/
Disallow: /user/login/
Disallow: /user/logout/
# Paths (no clean URLs)
Disallow: /?q=admin/
Disallow: /?q=comment/reply/
Disallow: /?q=filter/tips/
Disallow: /?q=node/add/
Disallow: /?q=search/
Disallow: /?q=user/password/
Disallow: /?q=user/register/
Disallow: /?q=user/login/
Disallow: /?q=user/logout/
```

We should also note that the recommended `crawl-delay` for this website is `10` seconds, and we should adhere to this as much as possible.

### Specifying the `User-Agent`

Aside from following the site’s `robots.txt` rules, it’s also good practice to identify your scraper when making requests by setting a clear `User-Agent` header. This lets website owners know who (or what) is accessing their site and helps your requests look like responsible traffic rather than anonymous or potentially malicious activity.

In [1]:
headers = {"User-Agent": "llorenzo-scraper/1.0(contact:llorenzo@aim.edu)"}

## 1 Performing HTTP requests using `requests`

`requests` is a simple HTTP library for Python which we will use to send client requests and interpret server responses programatically.

### General Usage

Let's look at some key features of the `requests` library.

In [2]:
from time import sleep

import requests

First, we demonstrate how we perform **GET** requests using `requests`.

In [3]:
url = 'http://pythonscraping.com/pages/page1.html'

response = requests.get(url, headers=headers)
sleep(10)

The above code generates a GET request from our client then the server response is collected into the variable `response`.

In [4]:
type(response)

requests.models.Response

The output is a `Response` object which has several useful attributes. First, we can check for the status code of the response:

In [5]:
response.status_code

200

There are several possible status codes, each of which describes the result of the requests. We list some of the important status codes below:

| Code | Description |
| :--- | :---------- |
| `200` | OK, request succeed |
| `404` | Not found, requested resource cannot be found |
| `403` | Access denied |
| `429` | Too many requests (rate-limited) |
| `500` | Internal server error |

If you encounter an HTTP status error, you’ll want to handle it gracefully so you can control how your program behaves when something goes wrong.

In practice, this usually means logging the error (for example, to a file) and deciding whether to retry, skip the request, or stop the program altogether. A common pattern you’ll see is using try–except blocks to catch and handle these errors cleanly.

In [7]:
url = 'http://pythonscraping.com/pages/page1337.html'
headers = {"User-Agent": "llorenzo-scraper/1.0(contact:llorenzo@aim.edu)"}

try:
    response = requests.get(url, headers=headers)
    sleep(10)
    response.raise_for_status()
except requests.RequestException as e:
    print("Failed to fetch page: ", e)

Failed to fetch page:  404 Client Error: Not Found for url: https://pythonscraping.com/pages/page1337.html


We will be using the above code multiple times throughout this notebook, thus, we might as well create a function for obtaining responses when we request for web pages.

In [8]:
def get_webpage(url, headers, sleep_time=10):
    """Perform GET request on a given url then collect the resulting
    response

    Parameters
    ----------
    url : str
        URL string of the website to be scraped
    headers : dict
        Set of key-value pairs to be passed as headers during the
        request
    sleep_time : numeric, default=10
        Delay time after the request

    Returns
    -------
    response : Response object
        Response of the server to the request
    """
    try:
        response = requests.get(url, headers=headers)
        sleep(sleep_time)
        response.raise_for_status()
    except requests.RequestException as e:
        print("Failed to fetch page: ", e)
    else:
        print(f"Page has been successfully fetched! {url}")

    return response

## 2 Web Scraping using `BeautifulSoup`

Now that we know how we can create requests from our client using Python, the next step is knowing how to parse the resulting HTML content and extract the data that we want from the website.

### Basic Usage of `BeautifulSoup`

Let's first get once again the `page1.html` from the `pythonscraping` website which contains a simple html file.

In [9]:
url = 'http://pythonscraping.com/pages/page1.html'
response = get_webpage(url, headers)

Page has been successfully fetched! http://pythonscraping.com/pages/page1.html


We can get the contents of the HTML file by calling the `content` attribute. Verify that the content is usually an HTML file. On usual requests you would get an HTML file, but there are websites also where the output can be of other format such as `xml` or `json` files.

In [10]:
response.content

b'<html>\n<head>\n<title>A Useful Page</title>\n</head>\n<body>\n<h1>An Interesting Title</h1>\n<div>\nLorem ipsum dolor sit amet, consectetur adipisicing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.\n</div>\n</body>\n</html>\n'

We can then use Beautiful Soup (`bs4`) to parse this HTML file. The most comonly used object in `bs4` is the `BeautifulSoup` object, which we import accordingly.

In [11]:
from bs4 import BeautifulSoup

When we use the `BeautifulSoup` object, we specify two parameters - the first is the HTML string object to parse, while the second specifies the parser that you want Beautiful Soup to use.

The currently supported parsers are `html.parser` (Python's built-in HTML parser), `lxml`, `html5lib`. Different parsers will create different parse trees from the same document. For more info please check the [documentation](https://beautiful-soup-4.readthedocs.io/en/latest/#differences-between-parsers).

If given a perfectly formed HTML, the data structure will be identical for different parsers. But if the HTML is not perfectly-formed, different parsers will give different results. Among the html parsers, the `html5lib` parser uses techniques that are part of the `HTML5` standard. Thus, this is what we'll be using on most of our examples.

In [12]:
soup = BeautifulSoup(response.content, 'html5lib')

Afterwards, the HTML will be parsed into an object with a hierarchical structure:

In [13]:
print(soup.prettify())

<html>
 <head>
  <title>
   A Useful Page
  </title>
 </head>
 <body>
  <h1>
   An Interesting Title
  </h1>
  <div>
   Lorem ipsum dolor sit amet, consectetur adipisicing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.
  </div>
 </body>
</html>



The `BeautifulSoup` object represents the HTML document as a whole, meanwhile, a `Tag` object corresponds to an HTML tag in the original document.

In [14]:
soup.html

<html><head>
<title>A Useful Page</title>
</head>
<body>
<h1>An Interesting Title</h1>
<div>
Lorem ipsum dolor sit amet, consectetur adipisicing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.
</div>


</body></html>

In [15]:
soup.h1

<h1>An Interesting Title</h1>

You can traverse the document in one direction by chaining method calls of corresponding children tags.

In [16]:
soup.html.body

<body>
<h1>An Interesting Title</h1>
<div>
Lorem ipsum dolor sit amet, consectetur adipisicing elit, sed do eiusmod tempor incididunt ut labore et dolore magna aliqua. Ut enim ad minim veniam, quis nostrud exercitation ullamco laboris nisi ut aliquip ex ea commodo consequat. Duis aute irure dolor in reprehenderit in voluptate velit esse cillum dolore eu fugiat nulla pariatur. Excepteur sint occaecat cupidatat non proident, sunt in culpa qui officia deserunt mollit anim id est laborum.
</div>


</body>

In [17]:
soup.html.body.h1

<h1>An Interesting Title</h1>

In [18]:
type(soup.html.body)

bs4.element.Tag

Each tag also has the `name` accessible as `.name`:

In [19]:
soup.html.body.name

'body'

In [20]:
soup.html.body.h1.name

'h1'

And you can extract the text within a tag using the `.text` attribute.

In [21]:
soup.html.body.h1.text

'An Interesting Title'

Or you can opt to use the `.get_text()` function for better readability.

In [23]:
soup.html.body.h1.get_text()

'An Interesting Title'

### Finding tags

To filter HTML pages and find only a list of desired elements, `BeautifulSoup` provides us with the `find()` and `find_all()` function.

- `find()` returns the first match. e.g. `soup.find('a')` returns the first HTML element with a `a` tag.
- `find_all()` returns a list of all matches. `soup.find_all('a')` returns an iterable of HTML elements with a `a` tag.

You can define filters as a string, regular expression, or a list, which matches the given input to the tag of the corresponding HTML element. The definition of the two functions is as follows:

```
find_all(tag, attrs, recursive, text, limit, **kwargs)
find(tag, attrs, recursive, text, **kwargs)
```

As an example of the usage of `find()` and `find_all()`, let's look at scraping the web page http://www.pythonscraping.com/pages/warandpeace.html

In [24]:
url = 'http://www.pythonscraping.com/pages/warandpeace.html'

response = get_webpage(url, headers)

Page has been successfully fetched! http://www.pythonscraping.com/pages/warandpeace.html


After successfully fetching the page, we proceed to create a `BeautifulSoup` object:

In [25]:
soup = BeautifulSoup(response.content, 'html5lib')

You'll notice that the spoken lines of the characters in the stories are highlighted in red, meanwhile the names of the characters are in green.

In [26]:
for line in soup.prettify().splitlines()[78: 85]:
    print(line)

     Heavens! what a virulent attack!
    </span>
    " replied
    <span class="green">
     the prince
    </span>
    , not in the


As such, we can extract all the character names by filtering all `span` elements with `class` having a value of `green`.

In [38]:
set([elem.text.replace('\n', ' ').lower() for elem in soup.find_all('span', {'class': ['green']})])

{'abbe morio',
 'anatole',
 'anna pavlovna',
 'anna pavlovna scherer',
 "anna pavlovna's",
 'baron funke',
 'dowager empress marya fedorovna',
 'empress marya fedorovna',
 'her majesty',
 'king of prussia',
 'le vicomte de mortemart',
 'montmorencys',
 'prince vasili',
 'prince vasili kuragin',
 'rohans',
 'st. petersburg',
 'the baron',
 'the emperor',
 'the empress',
 'the prince',
 'wintzingerode'}

### Navigating Trees

Recall that chaining tags enables you to navigate the `BeautifulSoup` tree in a single downward direction:

In [39]:
soup.body.div.p

<p>
It was in July, 1805, and the speaker was the well-known <span class="green">Anna
Pavlovna Scherer</span>, maid of honor and favorite of the <span class="green">Empress Marya
Fedorovna</span>. With these words she greeted <span class="green">Prince Vasili Kuragin</span>, a man
of high rank and importance, who was the first to arrive at her
reception. <span class="green">Anna Pavlovna</span> had had a cough for some days. She was, as
she said, suffering from la grippe; grippe being then a new word in
<span class="green">St. Petersburg</span>, used only by the elite.
</p>

If we want to get all the `<p>` tags within the first `<div>` tag in the document, or anything more complicated than the first tag within a certain level, we would need to use other methods to do so.

In the next subsection, we show how we can also traverse the tree further with more control using various attributes and functions. We'll demonstrate how to navigate the tree downards, upwards, and sideways.

#### Going down

If we want to get all the tags directly one below a certain tag, we can use the `.contents` or `.children` attribute.

In [42]:
soup.body.div.p.contents

['\nIt was in July, 1805, and the speaker was the well-known ',
 <span class="green">Anna
 Pavlovna Scherer</span>,
 ', maid of honor and favorite of the ',
 <span class="green">Empress Marya
 Fedorovna</span>,
 '. With these words she greeted ',
 <span class="green">Prince Vasili Kuragin</span>,
 ', a man\nof high rank and importance, who was the first to arrive at her\nreception. ',
 <span class="green">Anna Pavlovna</span>,
 ' had had a cough for some days. She was, as\nshe said, suffering from la grippe; grippe being then a new word in\n',
 <span class="green">St. Petersburg</span>,
 ', used only by the elite.\n']

In [44]:
list(soup.body.div.p.children)

['\nIt was in July, 1805, and the speaker was the well-known ',
 <span class="green">Anna
 Pavlovna Scherer</span>,
 ', maid of honor and favorite of the ',
 <span class="green">Empress Marya
 Fedorovna</span>,
 '. With these words she greeted ',
 <span class="green">Prince Vasili Kuragin</span>,
 ', a man\nof high rank and importance, who was the first to arrive at her\nreception. ',
 <span class="green">Anna Pavlovna</span>,
 ' had had a cough for some days. She was, as\nshe said, suffering from la grippe; grippe being then a new word in\n',
 <span class="green">St. Petersburg</span>,
 ', used only by the elite.\n']

If you want to go deeper than one level, you can use the `.descendants` function instead.

In [45]:
list(soup.body.div.p.descendants)

['\nIt was in July, 1805, and the speaker was the well-known ',
 <span class="green">Anna
 Pavlovna Scherer</span>,
 'Anna\nPavlovna Scherer',
 ', maid of honor and favorite of the ',
 <span class="green">Empress Marya
 Fedorovna</span>,
 'Empress Marya\nFedorovna',
 '. With these words she greeted ',
 <span class="green">Prince Vasili Kuragin</span>,
 'Prince Vasili Kuragin',
 ', a man\nof high rank and importance, who was the first to arrive at her\nreception. ',
 <span class="green">Anna Pavlovna</span>,
 'Anna Pavlovna',
 ' had had a cough for some days. She was, as\nshe said, suffering from la grippe; grippe being then a new word in\n',
 <span class="green">St. Petersburg</span>,
 'St. Petersburg',
 ', used only by the elite.\n']

In [46]:
len(list(soup.body.div.p.descendants))

16

In [47]:
len(list(soup.body.div.p.children))

11

If we only want to get only the strings or text within the text of the descendants of a tag, we can use the `.strings` method.

In [50]:
list(soup.body.div.p.strings)

['\nIt was in July, 1805, and the speaker was the well-known ',
 'Anna\nPavlovna Scherer',
 ', maid of honor and favorite of the ',
 'Empress Marya\nFedorovna',
 '. With these words she greeted ',
 'Prince Vasili Kuragin',
 ', a man\nof high rank and importance, who was the first to arrive at her\nreception. ',
 'Anna Pavlovna',
 ' had had a cough for some days. She was, as\nshe said, suffering from la grippe; grippe being then a new word in\n',
 'St. Petersburg',
 ', used only by the elite.\n']

In [51]:
type(list(soup.body.div.p.strings)[0])

bs4.element.NavigableString

In [54]:
list(list(soup.body.div.p.strings)[0].parents)

[<p>
 It was in July, 1805, and the speaker was the well-known <span class="green">Anna
 Pavlovna Scherer</span>, maid of honor and favorite of the <span class="green">Empress Marya
 Fedorovna</span>. With these words she greeted <span class="green">Prince Vasili Kuragin</span>, a man
 of high rank and importance, who was the first to arrive at her
 reception. <span class="green">Anna Pavlovna</span> had had a cough for some days. She was, as
 she said, suffering from la grippe; grippe being then a new word in
 <span class="green">St. Petersburg</span>, used only by the elite.
 </p>,
 <div id="text">
 "<span class="red">Well, Prince, so Genoa and Lucca are now just family estates of the
 Buonapartes. But I warn you, if you don't tell me that this means war,
 if you still try to defend the infamies and horrors perpetrated by
 that Antichrist- I really believe he is Antichrist- I will have
 nothing more to do with you and you are no longer my friend, no longer
 my 'faithful slave,' as yo

Or `.stripped_strings` if we want to delete the trailing and leading whitespaces in each strings.

In [55]:
list(soup.body.div.p.stripped_strings)

['It was in July, 1805, and the speaker was the well-known',
 'Anna\nPavlovna Scherer',
 ', maid of honor and favorite of the',
 'Empress Marya\nFedorovna',
 '. With these words she greeted',
 'Prince Vasili Kuragin',
 ', a man\nof high rank and importance, who was the first to arrive at her\nreception.',
 'Anna Pavlovna',
 'had had a cough for some days. She was, as\nshe said, suffering from la grippe; grippe being then a new word in',
 'St. Petersburg',
 ', used only by the elite.']

#### Going Up

You can also access an element's parent with the `.parent` attribute.

In [56]:
soup.body.div.p.span.parent

<p>
It was in July, 1805, and the speaker was the well-known <span class="green">Anna
Pavlovna Scherer</span>, maid of honor and favorite of the <span class="green">Empress Marya
Fedorovna</span>. With these words she greeted <span class="green">Prince Vasili Kuragin</span>, a man
of high rank and importance, who was the first to arrive at her
reception. <span class="green">Anna Pavlovna</span> had had a cough for some days. She was, as
she said, suffering from la grippe; grippe being then a new word in
<span class="green">St. Petersburg</span>, used only by the elite.
</p>

#### Going Sideways

We can also use the `.next_sibling` and `.previous_sibling` to navigate between page elements that are on the same level of the parse tree:

In [58]:
soup.div.p.next_sibling

<p>
All her invitations without exception, written in French, and
delivered by a scarlet-liveried footman that morning, ran as follows:
</p>

In [59]:
soup.div.p.previous_sibling

'"\n'

In [60]:
soup.div.p.next_sibling.previous_sibling

<p>
It was in July, 1805, and the speaker was the well-known <span class="green">Anna
Pavlovna Scherer</span>, maid of honor and favorite of the <span class="green">Empress Marya
Fedorovna</span>. With these words she greeted <span class="green">Prince Vasili Kuragin</span>, a man
of high rank and importance, who was the first to arrive at her
reception. <span class="green">Anna Pavlovna</span> had had a cough for some days. She was, as
she said, suffering from la grippe; grippe being then a new word in
<span class="green">St. Petersburg</span>, used only by the elite.
</p>

Or if we want to list all the tags within the same level of the current tag, we can use `.next_siblings` or `.previous_siblings`

In [63]:
list(soup.div.p.next_siblings)[:2]

[<p>
 All her invitations without exception, written in French, and
 delivered by a scarlet-liveried footman that morning, ran as follows:
 </p>,
 <p>
 "<span class="red">If you have nothing better to do, Count [or Prince], and if the
 prospect of spending an evening with a poor invalid is not too
 terrible, I shall be very charmed to see you tonight between 7 and 10-
 Annette Scherer.</span>"
 </p>]

<div class="alert alert-info" role="info">

  <b>Example 1: A Totally Normal Gifts Website</b>

  Retrieve and extract the item titles, description, cost, and image url of the gifts found from the website https://www.pythonscraping.com/pages/page3.html. Place the result in a `pandas` `DataFrame`.

</div>

In [64]:
url = 'https://www.pythonscraping.com/pages/page3.html'

response = get_webpage(url, headers)

Page has been successfully fetched! https://www.pythonscraping.com/pages/page3.html


In [65]:
soup = BeautifulSoup(response.content, 'html5lib')

In [66]:
print(soup.prettify())

<html>
 <head>
  <style>
   img{
	width:75px;
}
table{
	width:50%;
}
td{
	margin:10px;
	padding:10px;
}
.wrapper{
	width:800px;
}
.excitingNote{
	font-style:italic;
	font-weight:bold;
}
  </style>
 </head>
 <body>
  <div id="wrapper">
   <img src="../img/gifts/logo.jpg" style="float:left;"/>
   <h1>
    Totally Normal Gifts
   </h1>
   <div id="content">
    Here is a collection of totally normal, totally reasonable gifts that your friends are sure to love! Our collection is
hand-curated by well-paid, free-range Tibetan monks.
    <p>
     We haven't figured out how to make online shopping carts yet, but you can send us a check to:
     <br/>
     123 Main St.
     <br/>
     Abuja, Nigeria
     <br/>
     We will then send your totally amazing gift, pronto! Please include an extra $5.00 for gift wrapping.
    </p>
   </div>
   <table id="giftList">
    <tbody>
     <tr>
      <th>
       Item Title
      </th>
      <th>
       Description
      </th>
      <th>
       Cost
      </th

In [67]:
import pandas as pd

In [71]:
list(soup.tr.next_siblings)

['\n\n',
 <tr class="gift" id="gift1"><td>
 Vegetable Basket
 </td><td>
 This vegetable basket is the perfect gift for your health conscious (or overweight) friends!
 <span class="excitingNote">Now with super-colorful bell peppers!</span>
 </td><td>
 $15.00
 </td><td>
 <img src="../img/gifts/img1.jpg"/>
 </td></tr>,
 '\n\n',
 <tr class="gift" id="gift2"><td>
 Russian Nesting Dolls
 </td><td>
 Hand-painted by trained monkeys, these exquisite dolls are priceless! And by "priceless," we mean "extremely expensive"! <span class="excitingNote">8 entire dolls per set! Octuple the presents!</span>
 </td><td>
 $10,000.52
 </td><td>
 <img src="../img/gifts/img2.jpg"/>
 </td></tr>,
 '\n\n',
 <tr class="gift" id="gift3"><td>
 Fish Painting
 </td><td>
 If something seems fishy about this painting, it's because it's a fish! <span class="excitingNote">Also hand-painted by trained monkeys!</span>
 </td><td>
 $10,005.00
 </td><td>
 <img src="../img/gifts/img3.jpg"/>
 </td></tr>,
 '\n\n',
 <tr class="gi

In [72]:
soup.tr.find_all('tr')

[]

In [69]:
soup.tr.find_next_siblings('tr')

[<tr class="gift" id="gift1"><td>
 Vegetable Basket
 </td><td>
 This vegetable basket is the perfect gift for your health conscious (or overweight) friends!
 <span class="excitingNote">Now with super-colorful bell peppers!</span>
 </td><td>
 $15.00
 </td><td>
 <img src="../img/gifts/img1.jpg"/>
 </td></tr>,
 <tr class="gift" id="gift2"><td>
 Russian Nesting Dolls
 </td><td>
 Hand-painted by trained monkeys, these exquisite dolls are priceless! And by "priceless," we mean "extremely expensive"! <span class="excitingNote">8 entire dolls per set! Octuple the presents!</span>
 </td><td>
 $10,000.52
 </td><td>
 <img src="../img/gifts/img2.jpg"/>
 </td></tr>,
 <tr class="gift" id="gift3"><td>
 Fish Painting
 </td><td>
 If something seems fishy about this painting, it's because it's a fish! <span class="excitingNote">Also hand-painted by trained monkeys!</span>
 </td><td>
 $10,005.00
 </td><td>
 <img src="../img/gifts/img3.jpg"/>
 </td></tr>,
 <tr class="gift" id="gift4"><td>
 Dead Parrot
 </

In [79]:
row = [elem.text.strip() for elem in soup.tr.find_next_siblings('tr')[0].children]
row

['Vegetable Basket',
 'This vegetable basket is the perfect gift for your health conscious (or overweight) friends!\nNow with super-colorful bell peppers!',
 '$15.00',
 '']

In [84]:
row[-1] = list(soup.tr.find_next_siblings('tr')[0].children)[-1].img['src']

In [85]:
row

['Vegetable Basket',
 'This vegetable basket is the perfect gift for your health conscious (or overweight) friends!\nNow with super-colorful bell peppers!',
 '$15.00',
 '../img/gifts/img1.jpg']

In [88]:
gifts = []
for element in soup.tr.find_next_siblings('tr'):
    row = [elem.text.strip() for elem in element.children]
    row[-1] = list(element.children)[-1].img['src']
    gifts.append(row)
pd.DataFrame(gifts, columns=['item', 'description', 'cost', 'image_url'])

,item,description,cost,image_url
0,Vegetable Basket,This vegetable basket is the perfect gift for ...,$15.00,../img/gifts/img1.jpg
1,Russian Nesting Dolls,"Hand-painted by trained monkeys, these exquisi...","$10,000.52",../img/gifts/img2.jpg
2,Fish Painting,"If something seems fishy about this painting, ...","$10,005.00",../img/gifts/img3.jpg
3,Dead Parrot,This is an ex-parrot! Or maybe he's only resting?,$0.50,../img/gifts/img4.jpg
4,Mystery Box,"If you love suprises, this mystery box is for ...",$1.50,../img/gifts/img6.jpg


## 3 Advanced Data Extraction with `BeautifulSoup`

We have demonstrated the basic features of `BeautifulSoup` which involves finding and accessing tags and attributes. We build on these tools further by introducing CSS selectors for precise targetting then also showing how we can also extract multiple pages or even multiple sites.

### CSS Selectors

CSS selectors can select elements baed on tags, classes, IDs, or attributes. The basic CSS selector syntax is as follows:

- `tag`: Selects all `<tag>` elements
- `.class`: Selects elements with `class="class"`
- `#id`: Selects an element with `id="id"`
- `[attribute="value"]`: Selects an element with the specified attribute.

You can use CSS selectors using `BeautifulSoup` through the `select()` method.

As an example, we select specific elements on the previously scraped website.

In [89]:
soup.select('tr#gift2')

[<tr class="gift" id="gift2"><td>
 Russian Nesting Dolls
 </td><td>
 Hand-painted by trained monkeys, these exquisite dolls are priceless! And by "priceless," we mean "extremely expensive"! <span class="excitingNote">8 entire dolls per set! Octuple the presents!</span>
 </td><td>
 $10,000.52
 </td><td>
 <img src="../img/gifts/img2.jpg"/>
 </td></tr>]

<div class="alert alert-info" role="info">

  <b>Example 2: A Totally Normal Gifts Website</b>

  Retrieve and extract the item titles, description, cost, and image url of the gifts found from the website https://www.pythonscraping.com/pages/page3.html using **CSS selectors**. Place the result in a `pandas` `DataFrame`.

</div>

In [90]:
soup.select('tr.gift')

[<tr class="gift" id="gift1"><td>
 Vegetable Basket
 </td><td>
 This vegetable basket is the perfect gift for your health conscious (or overweight) friends!
 <span class="excitingNote">Now with super-colorful bell peppers!</span>
 </td><td>
 $15.00
 </td><td>
 <img src="../img/gifts/img1.jpg"/>
 </td></tr>,
 <tr class="gift" id="gift2"><td>
 Russian Nesting Dolls
 </td><td>
 Hand-painted by trained monkeys, these exquisite dolls are priceless! And by "priceless," we mean "extremely expensive"! <span class="excitingNote">8 entire dolls per set! Octuple the presents!</span>
 </td><td>
 $10,000.52
 </td><td>
 <img src="../img/gifts/img2.jpg"/>
 </td></tr>,
 <tr class="gift" id="gift3"><td>
 Fish Painting
 </td><td>
 If something seems fishy about this painting, it's because it's a fish! <span class="excitingNote">Also hand-painted by trained monkeys!</span>
 </td><td>
 $10,005.00
 </td><td>
 <img src="../img/gifts/img3.jpg"/>
 </td></tr>,
 <tr class="gift" id="gift4"><td>
 Dead Parrot
 </

In [95]:
row = [elem.text.strip() for elem in soup.select('tr.gift')[0].select('td')]
row

['Vegetable Basket',
 'This vegetable basket is the perfect gift for your health conscious (or overweight) friends!\nNow with super-colorful bell peppers!',
 '$15.00',
 '']

In [98]:
row[-1] = soup.select('tr.gift')[0].select_one('img')['src']
row

['Vegetable Basket',
 'This vegetable basket is the perfect gift for your health conscious (or overweight) friends!\nNow with super-colorful bell peppers!',
 '$15.00',
 '../img/gifts/img1.jpg']

In [100]:
gifts = []
for element in soup.select('tr.gift'):
    row = [elem.text.strip() for elem in element.select('td')]
    row[-1] = element.select_one('img')['src']
    gifts.append(row)
gifts = pd.DataFrame(gifts, columns=['item', 'description', 'cost', 'image_url'])

In [101]:
gifts

,item,description,cost,image_url
0,Vegetable Basket,This vegetable basket is the perfect gift for ...,$15.00,../img/gifts/img1.jpg
1,Russian Nesting Dolls,"Hand-painted by trained monkeys, these exquisi...","$10,000.52",../img/gifts/img2.jpg
2,Fish Painting,"If something seems fishy about this painting, ...","$10,005.00",../img/gifts/img3.jpg
3,Dead Parrot,This is an ex-parrot! Or maybe he's only resting?,$0.50,../img/gifts/img4.jpg
4,Mystery Box,"If you love suprises, this mystery box is for ...",$1.50,../img/gifts/img6.jpg


### Scraping Multiple Webpages or Websites

So far, our examples have focused on extracting data from static web pages. Now that we know how to locate and extract specific tags and attributes, we can extend this idea to collect all the URLs on a page and then scrape each newly discovered page in turn.

Programs that follow links and move from page to page in this way are called **web crawlers**, since they “crawl” across the web by traversing links between pages.

As an example, let's demonstrate how we can create our own ***Pokédex™*** by extracting multiple webpages from a pokemon website - https://pokemondb.net/

Looking at the `https://pokemondb.net/robots.txt` file, the website disallows bots only on specific pages:

```
User-agent: *
Disallow: /pokebase/search?
Disallow: /pokebase/revisions
Disallow: /pokebase/meta/search?
Disallow: /pokebase/meta/revisions
Disallow: /pokebase/rmt/search?
Disallow: /pokebase/rmt/revisions
Crawl-delay: 2
```

We also note the crawl delay value of `2` for the website.

Specifically, let's scrape and create a database for the **FireRed & LeafGreen** version of Pokemon.

In [102]:
url = 'https://pokemondb.net/pokedex/game/firered-leafgreen'
headers = {"User-Agent": "llorenzo-scraper/1.0(contact:llorenzo@aim.edu)"}

response = get_webpage(url, headers=headers, sleep_time=5)

Page has been successfully fetched! https://pokemondb.net/pokedex/game/firered-leafgreen


In [103]:
soup = BeautifulSoup(response.content, 'html5lib')

In [104]:
soup.select('div.infocard')

[<div class="infocard"><span class="infocard-lg-img"><a href="/pokedex/bulbasaur"><img alt="Bulbasaur" class="img-fixed img-sprite img-sprite-v6" loading="lazy" src="https://img.pokemondb.net/sprites/ruby-sapphire/normal/bulbasaur.png"/></a></span><span class="infocard-lg-data text-muted"><small>#001</small><br/> <a class="ent-name" href="/pokedex/bulbasaur">Bulbasaur</a><br/> <small><a class="itype grass" href="/type/grass">Grass</a> · <a class="itype poison" href="/type/poison">Poison</a></small></span></div>,
 <div class="infocard"><span class="infocard-lg-img"><a href="/pokedex/ivysaur"><img alt="Ivysaur" class="img-fixed img-sprite img-sprite-v6" loading="lazy" src="https://img.pokemondb.net/sprites/ruby-sapphire/normal/ivysaur.png"/></a></span><span class="infocard-lg-data text-muted"><small>#002</small><br/> <a class="ent-name" href="/pokedex/ivysaur">Ivysaur</a><br/> <small><a class="itype grass" href="/type/grass">Grass</a> · <a class="itype poison" href="/type/poison">Poison<

In [105]:
soup.select('div.infocard')[0]

<div class="infocard"><span class="infocard-lg-img"><a href="/pokedex/bulbasaur"><img alt="Bulbasaur" class="img-fixed img-sprite img-sprite-v6" loading="lazy" src="https://img.pokemondb.net/sprites/ruby-sapphire/normal/bulbasaur.png"/></a></span><span class="infocard-lg-data text-muted"><small>#001</small><br/> <a class="ent-name" href="/pokedex/bulbasaur">Bulbasaur</a><br/> <small><a class="itype grass" href="/type/grass">Grass</a> · <a class="itype poison" href="/type/poison">Poison</a></small></span></div>

In [109]:
soup.select('div.infocard')[0].select_one('a.ent-name').text

'Bulbasaur'

In [111]:
soup.select('div.infocard')[0].small.text

'#001'

In [115]:
','.join([itype.text for itype in soup.select('div.infocard')[0].select('a.itype')])

'Grass,Poison'

In [117]:
soup.select('div.infocard')[0].img['src']

'https://img.pokemondb.net/sprites/ruby-sapphire/normal/bulbasaur.png'

In [122]:
f'https://pokemondb.net{soup.select('div.infocard')[0].a['href']}'

'https://pokemondb.net/pokedex/bulbasaur'

In [126]:
pokemons = []
for elem in soup.select('div.infocard'):
    pokemons.append({
        'pokemon_id': elem.small.text,
        'name': elem.select_one('a.ent-name').text,
        'type': ','.join([itype.text for itype in elem.select('a.itype')]),
        'image_url': elem.img['src'],
        'pokedex_url': f'https://pokemondb.net{elem.a['href']}'
    })

In [127]:
pd.DataFrame(pokemons)

,pokemon_id,name,type,image_url,pokedex_url
0,#001,Bulbasaur,"Grass,Poison",https://img.pokemondb.net/sprites/ruby-sapphir...,https://pokemondb.net/pokedex/bulbasaur
1,#002,Ivysaur,"Grass,Poison",https://img.pokemondb.net/sprites/ruby-sapphir...,https://pokemondb.net/pokedex/ivysaur
2,#003,Venusaur,"Grass,Poison",https://img.pokemondb.net/sprites/ruby-sapphir...,https://pokemondb.net/pokedex/venusaur
3,#004,Charmander,Fire,https://img.pokemondb.net/sprites/ruby-sapphir...,https://pokemondb.net/pokedex/charmander
4,#005,Charmeleon,Fire,https://img.pokemondb.net/sprites/ruby-sapphir...,https://pokemondb.net/pokedex/charmeleon
...,...,...,...,...,...
146,#147,Dratini,Dragon,https://img.pokemondb.net/sprites/ruby-sapphir...,https://pokemondb.net/pokedex/dratini
147,#148,Dragonair,Dragon,https://img.pokemondb.net/sprites/ruby-sapphir...,https://pokemondb.net/pokedex/dragonair
148,#149,Dragonite,"Dragon,Flying",https://img.pokemondb.net/sprites/ruby-sapphir...,https://pokemondb.net/pokedex/dragonite
149,#150,Mewtwo,Psychic,https://img.pokemondb.net/sprites/ruby-sapphir...,https://pokemondb.net/pokedex/mewtwo


Let's populate our pokedex database further by scraping the height, weight, and base stats of the pokemon in our database.

In [ ]:
# This code runs for about 15 minutes
for i, pokemon in pokemons.iterrows():
    # Scrape pokedex webpage of current pokemon
    response = get_webpage(pokemon['pokedex_url'], headers, sleep_time=5)
    soup = BeautifulSoup(response.content, 'html5lib')

    # Get height and weight
    pokemon['height'] = soup.find('th', string='Height').next_sibling.next_sibling.text
    pokemon['weight'] = soup.find('th', string='Weight').next_sibling.next_sibling.text

    # Parse the stats
    stats = {}
    for stat_elem in (soup.find('h2', string='Base stats').next_sibling
                          .next_sibling.select_one('table.vitals-table')
                          .find_all('th', limit=6)):
        stats[stat_elem.text] = int(stat_elem.next_sibling.next_sibling.text)
    pokemon['stats'] = stats

    # Write to output file
    with open('outputs/pokedex.jsonl', 'a') as f:
        f.write(pokemon.to_json() + '\n')

<img src="images/banner-down.png" style="width: 100%;">